## Initialization

In [ ]:
# Imports

from math import exp
from itertools import starmap
from pathlib import Path
from typing import Callable, TypeVar, Any, Literal, overload
from functools import reduce
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import deconvolve

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.dataframe_validation import DetectorDataframeColumn
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.helpers import stop, get_input_with_default
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.types import BimodalBounds, BimodalParams
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_generation import (
    generate_nasa_neutron_window,
    generate_n_distro_neutron_window
)
from data_processing.types import NasaGenerationSettings, WindowType, NeutronWindowSettings
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers import (
    stop,
    get_input_with_default,
    input_experiment_ids,
    get_midpoints_from_min_max_series
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.processing.neutron_window_strategy.strategy_factory import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import AbstractNeutronStrategy

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

## Functions

In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

## Run Settings

In [ ]:
exp_id = "ID-493"

phd_data_path = Path() / "pulse_height_distribution"
phd_input_path = phd_data_path / "input"
phd_output_path = phd_data_path / "output"
exp_data_filename = f"{exp_id}-n_spectrum.csv"
exp_data_path = phd_input_path / exp_data_filename

sim_data_filenames = [
    # "output_run1.txt",
    # "output_run2.txt",
    # "output_run3.txt",
    # "output_run4.txt",
    # "output_run5.txt",
    # "output_run6.txt",
    # "output_run7.txt",
    # "output_run8.txt",
    # "output_run9.txt",
    # "output_run10.txt",
    # "output-2024-07-16.txt",
    # "sigma-0.3-output.txt",
    # "output.txt",
    "output_sigma_0.5MeV_wresolution.txt",
    # "output_14.1MeV.txt",
    # "output_14.1MeV_1e4.txt",
    # "output_14.1MeV_1e5.txt",
    # "output_4MeV_1e5.txt",
    # "output_8MeV_1e5.txt"
]
# sim_data_description = "0.5MeV variability"
# sim_data_description = "July 9 simulation"
sim_data_description = "sigma 0.5 fixing plot y limits"
sim_data_combine = False
sim_data_paths = {filename: phd_input_path / filename
                  for filename in sim_data_filenames}

## Loading

### Experimental Data

In [ ]:
exp_df = pd.read_csv(
    exp_data_path,
    usecols=[
        "Light output bin start (MeVee)",
        "Light output bin end (MeVee)",
        "Neutron counts"
    ]
)
exp_df['Count error'] = exp_df['Neutron counts'].pow(1./2)

In [ ]:
midpoints = get_midpoints_from_min_max_series(
    exp_df["Light output bin start (MeVee)"],
    exp_df["Light output bin end (MeVee)"],
    exp_df.index
)
exp_df["Light output (MeVee)"] = midpoints

### Simulation Data

In [ ]:
sim_dfs = {}
for sim_filename, sim_data_path in sim_data_paths.items():
    if sim_filename in ["output_run1.txt", "output_run2.txt"]:
        sim_df = pd.read_csv(sim_data_path, sep="\t", index_col=False)
    else:
        sim_df = pd.read_fwf(sim_data_path)
    sim_df = sim_df[["NPS", "det_pulse (MeVee)"]].copy()
    sim_df.columns = ["Count rate", "Neutron light output (MeVee)"]
    sim_dfs[sim_filename] = sim_df

In [ ]:
sim_l_cut_dfs = {}
bins_lo = exp_df["Light output bin start (MeVee)"]
bins_hi = exp_df["Light output bin end (MeVee)"]
bins = pd.IntervalIndex.from_arrays(bins_lo, bins_hi)
for sim_filename, sim_df in sim_dfs.items():
    light_output_cut = pd.cut(sim_df["Neutron light output (MeVee)"],
                              bins=bins)
    sim_l_cut_dfs[sim_filename] = light_output_cut

In [ ]:
binned_sim_dfs = {}
for sim_filename, sim_l_cut_df in sim_l_cut_dfs.items():
    binned_sim_df = sim_df.groupby(sim_l_cut_df).sum()[["Count rate"]].copy()
    binned_sim_energy_bins = binned_sim_df.index.to_series()
    midpoints = binned_sim_energy_bins.apply(lambda x: x.mid)
    binned_sim_df["Light output (MeVee)"] = midpoints
    binned_sim_dfs[sim_filename] = binned_sim_df

In [ ]:
if sim_data_combine:
    count_series = []
    light_output_series = None
    for sim_filename, binned_sim_df in binned_sim_dfs.items():
        count_series.append(binned_sim_df['Count rate'])
        if light_output_series is None:
            light_output_series = binned_sim_df["Light output (MeVee)"]
    combined_series = pd.concat(count_series, axis=1)
    combined_mean = combined_series.mean(axis=1)
    combined_sd = combined_series.std(axis=1)
    combined_stats_df = pd.DataFrame(
        {
            "Light output (MeVee)": light_output_series,
            "Count rate": combined_mean,
            "Rate error": combined_sd
        }
    )
    binned_sim_dfs = {"combined": combined_stats_df}
else:
    for binned_sim_df in binned_sim_dfs.values():
        binned_sim_df['Rate error'] = binned_sim_df['Count rate'].pow(1./2)

## Normalization

In [ ]:
counts = exp_df['Neutron counts']
errors = exp_df['Count error']

exp_max = counts.max()
exp_max_idx = counts.idxmax()
exp_max_error = errors.loc[exp_max_idx]
rel_square_max_error = (exp_max_error / exp_max) ** 2

# norm_counts = counts / exp_max
norm_counts = counts
exp_df['Counts (normalized)'] = norm_counts

# rel_norm_errors = relative_rmse(counts, errors, exp_max, exp_max_error)
# norm_errors = rel_norm_errors * norm_counts
norm_errors = errors
exp_df['Error (normalized)'] = norm_errors

In [ ]:
sim_max = 0
sim_max_error = 0
for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    errors = binned_sim_df['Rate error']
    this_sim_max = counts.max()
    this_sim_max_idx = counts.idxmax()
    this_sim_max_error = errors[this_sim_max_idx]
    if this_sim_max > sim_max:
        sim_max = this_sim_max
        sim_max_error = this_sim_max_error
sim_factor = exp_max / sim_max
# sim_factor_error = relative_rmse(exp_max, exp_max_error, sim_max, sim_max_error)
# rel_square_max_error = (sim_max_error / sim_max) ** 2

for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    errors = binned_sim_df['Rate error']
    
    norm_counts = counts * sim_factor
    binned_sim_df['Counts (normalized)'] = norm_counts
    
    # TODO normalize error
    # rel_square_errors = (errors / counts).pow(2).fillna(0)
    # rel_norm_errors = (rel_square_errors + rel_square_max_error).pow(1./2)
    # rel_norm_errors = relative_rmse(counts, errors, sim_factor, sim_factor_error)
    rel_norm_errors = relative_rmse([(counts, errors), (exp_max, exp_max_error), (sim_max, sim_max_error)])
    norm_errors = rel_norm_errors * norm_counts
    binned_sim_df['Error (normalized)'] = norm_errors

## Plotting

In [ ]:
sim_xys = {}
sim_errors = {}
sim_xys_raw = {}
sim_errors_raw = {}
for sim_filename, binned_sim_df in binned_sim_dfs.items():
    sim_x = binned_sim_df["Light output (MeVee)"].astype(float) * 1000
    sim_y = binned_sim_df["Counts (normalized)"].astype(float)
    sim_raw_y = binned_sim_df["Count rate"].astype(float)
    sim_error = binned_sim_df["Error (normalized)"].astype(float)
    sim_raw_error = binned_sim_df["Rate error"].astype(float)
    
    sim_xys[sim_filename] = (sim_x, sim_y)
    sim_xys_raw[sim_filename] = (sim_x, sim_raw_y)
    sim_errors[sim_filename] = sim_error
    sim_errors_raw[sim_filename] = sim_raw_error

exp_x = exp_df["Light output (MeVee)"] * 1000
exp_y = exp_df["Counts (normalized)"]
exp_error = exp_df["Error (normalized)"]
exp_raw_y = exp_df["Neutron counts"]
exp_raw_y_error = exp_df["Count error"]

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(8, 5))
# fig.tight_layout()

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    sim_error = sim_errors.get(sim_filename)
    axs.errorbar(
        sim_x,
        sim_y,
        yerr=sim_error,
        ls='-',
        marker='',
        ms=5,
        capsize=2,
        # color='#424242FF',
        label="Simulation"
    )
axs.errorbar(
    exp_x,
    exp_y,
    yerr=exp_error,
    ls='--',
    marker='',
    ms=5,
    capsize=2,
    label="Experiment"
)

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_xlim(200, 240)
# axs.set_ylim(0.99, 1.01)
axs.set_ylabel('Normalized counts', fontsize=14)
# axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=12)
# axs.annotate(
#     'Experiment',
#     (700, 0.15),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14
# )
# axs.annotate(
#     'Simulation',
#     (400, 0.05),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend(
    # loc=(0.78, 0.85)
)
fig.tight_layout()

base_file_name = f"PHD Sim {sim_data_description} vs Exp {exp_id}"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
fig, axs = plt.subplots(
    1, 1,
    figsize=(8, 5)
)

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    sim_error = sim_errors.get(sim_filename)
    axs.errorbar(
        sim_x,
        sim_y,
        yerr=sim_error,
        ls='-',
        marker='',
        ms=5,
        capsize=2,
        # color='#424242FF',
        label="Simulation"
    )
axs.errorbar(
    exp_x,
    exp_y,
    yerr=exp_error,
    ls='--',
    marker='',
    ms=5,
    capsize=2,
    label="Experiment"
)

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
axs.set_ylim(bottom=1)
# axs.set_ylim(-5,180)
axs.set_ylabel('Normalized counts', fontsize=14)
axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=12)
# axs.annotate(
#     'Experiment',
#     (700,0.15),
#     xytext = None,
#     xycoords = 'data',
#     textcoords = 'data',
#     fontsize = 14
# )
# axs.annotate(
#     'Simulation',
#     (300, 0.0),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend(
    # loc=(0.8, 0.85)
)
fig.tight_layout()

base_file_name = f"PHD Sim {sim_data_description} vs Exp {exp_id} Log Scale"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()